In [2]:
from typing import TypedDict, Optional, List
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain_core.tools import tool
from dotenv import load_dotenv
from datetime import date


load_dotenv()

True

In [3]:
class GeneralInformation(TypedDict):
    name: Optional[str]
    gender: Optional[str]
    date_of_birth: Optional[str]
    address: Optional[str]
    postal_code: Optional[str]
    email: Optional[str]
    phone: Optional[str]
    has_family_doctor: Optional[bool]
    family_doctor_name: Optional[str]
    has_psychiatrist: Optional[bool]
    psychiatrist_name: Optional[str]

class CurrentConcerns(TypedDict):
    primary_problems: Optional[str]
    goals: Optional[str]

class AliceState(TypedDict):
    messages: List[dict]
    current_section: Optional[str]
    sections_completed: List[str]
    safety_flag: bool
    conversation_started: bool
    general_information: GeneralInformation
    current_concerns: CurrentConcerns


llm = ChatOpenAI(model="gpt-4o-mini")



In [4]:
@tool
def save_answer(field_name: str, value: str) -> str:
    """
    Call this every time the patient answers a question or corrects a previous answer.
    field_name: the exact field name (e.g. 'name', 'gender', 'age', 'address')
    value: exactly what the patient said
    """
    return "saved"


In [5]:
@tool
def raise_safety_flag() -> str:
    """
    Call this immediately if the patient mentions:
    - Thoughts of suicide or self-harm
    - Thoughts of harming others
    - A current crisis situation
    """
    return "flag_raised"



In [6]:
tools = [save_answer, raise_safety_flag]
model    = llm.bind_tools(tools)


In [7]:
STATIC_PROMPT = """
You are ALICE, a warm and professional clinical intake assistant
at St. Michael's Hospital, Unity Health Toronto.

Your role:
- Help collect clinical intake information before the patient's appointment.
- Collect information one question at a time.
- Speak calmly, kindly, and professionally.
- Do not diagnose, treat, or give medical advice.
- Do not guess missing information.
- Do not ask for information that has already been collected.
- If an answer is unclear, ask a brief clarification.
- If the patient corrects previous information, save the corrected value.

Conversation rules:
- Ask only one intake question at a time.
- Keep replies short and clear.

Small-talk and orientation rules:
- If the patient seems nervous or confused, reassure them briefly.
- If the patient asks whether they must answer, say they can skip anything they are not comfortable answering.

Tool rules:
- Call save_answer only when the patient gives a clear intake answer.
- Call save_answer when the patient corrects a previous intake answer.
- If the patient gives information for a different intake field, save it to the correct field, then return to the current required question.
- If the patient answers multiple fields in one message, call save_answer separately for each one, then ask for the next pending field.
- Call raise_safety_flag according to the safety protocol below.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SAFETY PROTOCOL — READ CAREFULLY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

STEP 1 — DETECT DISTRESS SIGNALS

Before every response, ask yourself:
  "Did the patient express any negative emotion, frustration, tension,
   or difficulty — however mild?"

If yes → pause intake and go to Step 2. Always.

You do not need to see an obvious danger sign.
A passing complaint, a frustrated comment about someone, a vague remark
about stress, or a brief expression of sadness is enough to trigger a
check-in. Your job is not to assess severity — it is to notice and ask.
It is always better to check than to assume everything is fine.

STEP 2 — PROBE

Acknowledge what the patient said warmly and briefly.
Then ask one natural question that checks whether they are safe —
directed at themselves, at someone else, or both, depending on what
they said.

Do not use a scripted question. Let the probe follow naturally from
their words. The question should feel like a genuine human concern,
not a clinical checkbox.

Ask the probe ONCE. Do not ask multiple safety questions in a row.

STEP 3 — EVALUATE THE RESPONSE AND DECIDE

After the patient answers your probe, apply this rule:

  RAISE THE FLAG if the response is anything other than a clear,
  confident denial. This includes:
    - Yes, maybe, sometimes, I don't know, possibly, sort of, I guess
    - "I've thought about it", "I'm not sure"
    - Evasion — changing the subject, "I don't want to talk about it",
      going quiet, or deflecting without answering
    - Any answer that leaves doubt

  DO NOT RAISE THE FLAG only if the patient gives a clear, confident no:
    - "No", "No, nothing like that", "I'm fine, just stressed"
    Acknowledge briefly and return to intake.

  WHEN IN DOUBT → RAISE THE FLAG.
  It is always safer to flag than to miss a risk.

STEP 4 — AFTER RAISING THE FLAG
- Call raise_safety_flag immediately.
- Do not write anything else in that turn.
- The system will display the safety message automatically.
- On the next turn, resume intake calmly with the current pending question.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SAFETY EXAMPLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

These examples show the pattern. Real conversations will vary — use
your judgment, not just these cases.

Example 1 — Mild frustration about another person → probe → clear no → continue:
  Patient:   "My father is so annoying."
  ALICE:     "That sounds frustrating. Sometimes when things are tense at
              home it can really weigh on you — has it ever gotten to the
              point where you've felt unsafe or had thoughts of hurting
              yourself or someone else?"
  Patient:   "No no, he's just loud."
  Action:    Acknowledge briefly, return to intake. Do NOT flag.

Example 2 — Vague hopelessness → probe → vague answer → flag:
  Patient:   "I just feel like I can't keep going."
  ALICE:     "I'm really sorry you're feeling that way. I want to make
              sure you're safe — have you had any thoughts of hurting
              yourself?"
  Patient:   "I don't know... maybe."
  Action:    call raise_safety_flag()

Example 3 — Anger toward another person → probe → evasive → flag:
  Patient:   "I'm so angry at my brother I don't know what I might do."
  ALICE:     "That sounds like a really difficult situation. I want to make
              sure everyone is safe — have you had any thoughts of hurting
              him or anyone else?"
  Patient:   "I don't really want to talk about that."
  Action:    call raise_safety_flag()

Example 4 — Explicit self-harm or harm to others → skip probe → flag immediately:
  Patient:   "I've been thinking about suicide." / "I want to hurt my roommate."
  Action:    call raise_safety_flag() immediately. No probe needed.

Example 5 — Fear of another person → probe → clear no → continue:
  Patient:   "My partner has a bad temper and it scares me."
  ALICE:     "That sounds really hard. Are you safe right now — is anyone
              hurting or threatening you?"
  Patient:   "No, it's not physical, I'm okay."
  Action:    Acknowledge briefly, return to intake. Do NOT flag.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
IMPORTANT SAFETY CONSTRAINTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- Ask the probe ONCE only. Do not repeat or rephrase if the patient deflects.
- Do not ask multiple safety questions back to back.
- Do not provide therapy, advice, or diagnosis.
- Do not try to de-escalate or solve the patient's problem yourself.
- After flagging, do not linger on the topic — resume intake on the next turn.


IMPORTANT EXCEPTION:
Physical health symptoms mentioned during intake are clinical information,
not distress signals. Never trigger the safety probe for physical complaints.

Examples that are NOT distress signals:
- "I have a headache"
- "my back has been hurting"
- "I feel tired all the time"
- "I have chest pain"
- "I haven't been sleeping well"

These are symptoms the patient is reporting as part of their intake.
Treat them as answers to primary_problems, not as safety concerns.

"""



In [8]:
REQUIRED_FIELDS = {
    "general_information": [
        "name",
        "gender",
        "date_of_birth",
        "address",
        "postal_code",
        "email",
        "phone",
        "has_family_doctor",
        "family_doctor_name",
        "has_psychiatrist",
        "psychiatrist_name",
    ],
    "current_concerns": [
        "primary_problems",
        "goals",
    ],
}

In [9]:
import re
from datetime import datetime, date



def validate_date_of_birth(value: str):
    try:
        dob = datetime.strptime(value.strip(), "%d/%m/%Y").date()
        if not (date(1900, 1, 1) <= dob < date.today()):
            return False, "Please enter a valid date of birth."
        return True, dob.strftime("%d/%m/%Y")
    except ValueError:
        return False, "Invalid format. Please use DD/MM/YYYY."


def validate_postal_code(value: str):
    value = value.strip().upper().replace(" ", "")
    if re.match(r'^[A-Z]\d[A-Z]\d[A-Z]\d$', value):
        return True, value
    return False, "Invalid postal code. Please use the format A#A#A#, for example K2C2K9."


def validate_email(value: str):
    value = value.strip()
    if "@" in value and "." in value.split("@")[-1]:
        return True, value
    return False, "Invalid email address."

def validate_address(value: str):
    province_map = {
        "ON": "Ontario",
        "BC": "British Columbia",
        "AB": "Alberta",
        "SK": "Saskatchewan",
        "MB": "Manitoba",
        "QC": "Quebec",
        "NB": "New Brunswick",
        "NS": "Nova Scotia",
        "PE": "Prince Edward Island",
        "NL": "Newfoundland and Labrador",
        "YT": "Yukon",
        "NT": "Northwest Territories",
        "NU": "Nunavut",
    }
    province_names = {v.lower(): v for v in province_map.values()}

    value = value.strip()
    if not value:
        return False, "Please provide a complete address."

    parts = [p.strip() for p in value.split(",")]

    if len(parts) < 2:
        return False, "Please include street, city, and province."

    formatted = []
    for i, part in enumerate(parts):
        if i == len(parts) - 1:
            key_upper = part.strip().upper()
            key_lower = part.strip().lower()
            if key_upper in province_map:
                formatted.append(province_map[key_upper])
            elif key_lower in province_names:
                formatted.append(province_names[key_lower])
            else:
                formatted.append(part.strip().title())
        else:
            formatted.append(part.strip().title())

    return True, ", ".join(formatted)

def validate_phone(value: str):
    digits = re.sub(r'\D', '', value)
    if 10 <= len(digits) <= 15:
        return True, digits
    return False, "Phone number must be 10 digits (or include country code)."

def validate_boolean(value: str):
    v = value.lower().strip()
    if v.startswith(("yes", "y", "true")):
        return True, True
    if v.startswith(("no", "n", "false")):
        return True, False
    return False, "Please answer yes or no."

def validate_doctor_name(value: str):
    if value.strip().lower() == "unknown":
        return True, "unknown"
    return True, value.strip().upper()


FIELD_VALIDATORS = {
    "date_of_birth":      validate_date_of_birth,
    "postal_code":        validate_postal_code,
    "email":              validate_email,
    "phone":              validate_phone,
    "has_family_doctor":  validate_boolean,
    "has_psychiatrist":   validate_boolean,
    "family_doctor_name": validate_doctor_name,
    "psychiatrist_name":  validate_doctor_name,
    "validate_address" :validate_address,
}

In [10]:
SECTION_PROMPTS = {
    "general_information": """
Collect the following fields in order:
name, gender, date_of_birth, address, postal_code, email, phone,
has_family_doctor, family_doctor_name, has_psychiatrist, psychiatrist_name

Name:
- Collect the patient's full legal name as it appears on their legal documents.
- Save in UPPERCASE.
- If the patient gives only one word, ask: "Is that your complete legal name?"
- If confirmed, save it. If not, ask for the missing part.

Gender:
- Ask how the patient would like their gender recorded.
- Do not assume gender from their name.
- Save the whole answe with the first letter in UPPERCASE exactly as the patient states it.

Date of birth:
- Ask the patient for their date of birth in DD/MM/YYYY format.
- If the patient gives it in DD/MM/YYYY already, save it as is.
- If the patient gives it in a written format like "March 4th 1990" or 
  "4 March 1990", convert it to DD/MM/YYYY before saving.
- Example: "March 4th 1990" → save_answer("date_of_birth", "04/03/1990")
- Example: "4 March 1990"   → save_answer("date_of_birth", "04/03/1990")
- Example: "04/03/1990"     → save_answer("date_of_birth", "04/03/1990")
- If the date is completely unclear, ask the patient to repeat it in DD/MM/YYYY.

Address:
- Collect the patient's full legal home address.
- It must include all three parts: street address, city, and province.
- Do not save until you have all three parts.
- Save in this exact format: [Street Address], [City], [Province]
- Example: 123 Main Street, Toronto, Ontario

Postal code:
- Collect the patient's Canadian postal code.
- If it looks wrong, ask the patient to repeat it.

Email:
- Collect the patient's email address.
- If it does not look valid, ask them to repeat it.

Phone:
- Collect the patient's phone number.
- If it seems too short or unclear, ask them to repeat it.

Family doctor:
- Ask whether the patient currently has a family doctor.
- As soon as the patient answers, call save_answer("has_family_doctor", "yes") or save_answer("has_family_doctor", "no").
- Always save "yes" or "no" — nothing else.
- If yes, then ask for the doctor's full name including first and last name.
- If the patient gives only a last name, ask for the first name too.
- Save the name in UPPERCASE.
- If the patient does not know the name, save "unknown".
- If no, skip the family doctor name and move on.

Psychiatrist:
- Ask whether the patient currently has a psychiatrist.
- As soon as the patient answers, call save_answer("has_psychiatrist", "yes") or save_answer("has_psychiatrist", "no").
- Always save "yes" or "no" — nothing else.
- If yes, then ask for the psychiatrist's full name including first and last name.
- If the patient gives only a last name, ask for the first name too.
- Save the name in UPPERCASE.
- If the patient does not know the name, save "unknown".
- If no, skip the psychiatrist name and move on."""
,

"current_concerns": """
Collect the following fields in order:
primary_problems, goals

General rules:
- Let the patient answer in their own words.
- Do not diagnose, interpret, or provide therapy.
- If an answer is unclear, ask one short clarification.
- Always call save_answer as soon as the patient gives a clear response.
- If the patient adds or changes information after a field has already been saved,
  treat it as a correction.
- Fix spelling and structure, then call save_answer with the updated value.
- Do not re-ask the question — just save the correction and confirm it was updated.

Primary problems:
- Ask what main problems or difficulties the patient has been experiencing lately.
- If the patient says they have no problems or nothing is wrong:
  call save_answer("primary_problems", "No primary concerns reported at this time.")
- If the patient describes their problems:
  fix any spelling mistakes and improve sentence structure and clarity,
  then call save_answer("primary_problems", <cleaned version of their answer>).
  Keep their meaning exactly as intended — only fix the language, not the content.

Goals:
- Ask what the patient hopes to get out of their appointment.
- If the patient says they have no goals or nothing specific:
  call save_answer("goals", "No specific goals identified for this appointment.")
- If the patient describes their goals:
  fix any spelling mistakes and improve sentence structure and clarity,
  then call save_answer("goals", <cleaned version of their answer>).
  Keep their meaning exactly as intended — only fix the language, not the content.
""",
}

In [11]:
def get_pending_fields(state: AliceState, section: str) -> list:
    section_data = state[section]
    return [
        f for f in REQUIRED_FIELDS[section]
        if section_data.get(f) is None
        and not (f == "family_doctor_name" and section_data.get("has_family_doctor") is False)
        and not (f == "psychiatrist_name" and section_data.get("has_psychiatrist") is False)
    ]

In [12]:
def build_section_prompt(state: AliceState) -> str:
    today        = date.today().strftime("%B %d, %Y")
    name         = state["general_information"].get("name")
    section      = state["current_section"]
    pending      = get_pending_fields(state, section)
    next_field   = pending[0] if pending else None

    if not state["conversation_started"]:
        greeting = "This is the start. Greet the patient and introduce yourself as ALICE."
    else:
        greeting = "You have already introduced yourself. Do not greet again."

    return (
        f"Today's date: {today}\n"
        f"Patient name: {name if name else 'not collected yet'}\n"
        f"{greeting}\n\n"
        f"Next field to collect: {next_field}\n"
        f"All pending fields: {pending}\n\n"
        f"{SECTION_PROMPTS[section]}\n\n"
        "When all fields are collected say: "
        "'All done! Please take a look at the panel on the right — "
        "if anything looks incorrect or if you'd like to add or change anything, just let me know.'"
    )

In [13]:
def build_system_prompt(state: AliceState) -> str:
    return STATIC_PROMPT + "\n\n" + build_section_prompt(state)

In [19]:
import json
import os
from datetime import datetime

def save_record(state: AliceState):
    os.makedirs("records", exist_ok=True)

    name     = state["general_information"].get("name", "unknown")
    dob      = state["general_information"].get("date_of_birth", "unknown")
    name_slug = name.replace(" ", "_").lower()
    dob_slug  = dob.replace("/", "-")
    filename  = f"records/{name_slug}_{dob_slug}.json"

    record = {
        "saved_at":            datetime.now().isoformat(),
        "general_information": state["general_information"],
        "current_concerns":    state["current_concerns"],
        "safety_flag":         state["safety_flag"],
    }

    with open(filename, "w") as f:
        json.dump(record, f, indent=2, default=str)

In [20]:
def section_node(state: AliceState) -> AliceState:
    section = state["current_section"]
    section_data = state[section]                  
    model   = ChatOpenAI(model="gpt-4o-mini").bind_tools(tools)
    system  = {"role": "system", "content": build_system_prompt(state)}

    # ── Call 1: let the model fire tool calls ──────────────────────────────
    response = model.invoke([system] + state["messages"])

    safety_raised = False
    tool_results  = {}

    for tc in response.tool_calls:
        if tc["name"] == "save_answer":
            field     = tc["args"]["field_name"]
            value     = tc["args"]["value"]
            validator = FIELD_VALIDATORS.get(field)

            if validator:
                valid, result = validator(value)
                if valid:
                    state[section][field]  = result
                    tool_results[tc["id"]] = "saved"
                else:
                    tool_results[tc["id"]] = f"invalid: {result}"
            else:
                state[section][field]      = value
                tool_results[tc["id"]]     = "saved"

        elif tc["name"] == "raise_safety_flag":
            state["safety_flag"]           = True
            safety_raised                  = True
            tool_results[tc["id"]]         = "flag_raised"

    # ── Build reply — three possible situations ────────────────────────────

    if not response.tool_calls:
        # No tools fired — call 1 reply is the final reply
        reply         = response.content
        tool_exchange = []

    elif safety_raised:
        # Safety protocol — sanitise history, hardcoded reply
        state["messages"][-1] = {"role": "user", "content": "safety flag was raised and handled."}
        reply = (
            "I'm really glad you shared that with me. "
            "You are not alone. Please call 988 or go to your nearest emergency room. "
            "A member of our care team has been notified. "
            "Whenever you're ready, we can continue with the intake."
        )
        tool_exchange = []

    else:
        # Tools fired — call 2 to generate next question
        tool_exchange = [response] + [
            {"role": "tool", "content": tool_results[tc["id"]], "tool_call_id": tc["id"]}
            for tc in response.tool_calls
        ]
    updated_system = {"role": "system", "content": build_system_prompt(state)}
    reply = llm.invoke([updated_system] + state["messages"] + tool_exchange).content
    
    # ── Check section completion ───────────────────────────────────────────
    pending = get_pending_fields(state, section)
    if not pending and section not in state["sections_completed"]:
        state["sections_completed"].append(section)
        
    # Auto-save when both sections are done
    if set(state["sections_completed"]) == {"general_information", "current_concerns"}:
        save_record(state)


    state["messages"]             = state["messages"] + tool_exchange + [{"role": "assistant", "content": reply}]
    state["conversation_started"] = True
    return state

In [21]:
builder = StateGraph(AliceState)
builder.add_node("section", section_node)
builder.set_entry_point("section")
builder.add_edge("section", END)

graph = builder.compile()


In [22]:
initial_state: AliceState = {
    "messages": [],
    "current_section": None,
    "sections_completed": [],
    "safety_flag": False,
    "conversation_started": False,
    "general_information": {
        "name": None,
        "gender": None,
        "date_of_birth": None,
        "postal_code":None,
        "address": None,
        "email": None,
        "phone": None,
        "has_family_doctor": None,
        "family_doctor_name": None,
        "has_psychiatrist": None,
        "psychiatrist_name": None,
    },   
    "current_concerns": {
        "primary_problems": None,
        "goals": None,
    },
}



In [23]:
import gradio as gr
import copy

from PIL import Image, ImageDraw
import tempfile, os

def make_avatar(color, size=80):
    img  = Image.new('RGBA', (size, size), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)
    draw.ellipse([0, 0, size - 1, size - 1], fill=color)
    path = os.path.join(tempfile.gettempdir(), f"avatar_{color[0]}.png")
    img.save(path)
    return path

ALICE_AVATAR = "https://img.icons8.com/color/96/nurse-female.png"
USER_AVATAR  = make_avatar((99, 115, 129))
ALICE_AVATAR = "https://img.icons8.com/color/96/nurse-female.png"
# USER_AVATAR  = "https://img.icons8.com/color/96/user-circle-skin-type-3.png"

FIELD_LABELS = {
    "general_information": {
        "name":               "Full Name",
        "gender":             "Gender",
        "date_of_birth":      "Date of Birth",
        "address":            "Address",
        "postal_code":        "Postal Code",
        "email":              "Email",
        "phone":              "Phone",
        "has_family_doctor":  "Has Family Doctor",
        "family_doctor_name": "Family Doctor Name",
        "has_psychiatrist":   "Has Psychiatrist",
        "psychiatrist_name":  "Psychiatrist Name",
    },
    "current_concerns": {
        "primary_problems": "Primary Problems",
        "goals":            "Goals",
    }
}

def format_display(alice_state):
    if not alice_state or not alice_state.get("current_section"):
        return """
        <div style="font-family:'Inter',sans-serif;padding:20px;
                    text-align:center;color:#94a3b8;font-size:13px;font-style:italic;">
            Select a section above to begin
        </div>
        """

    section = alice_state["current_section"]
    data    = alice_state[section]
    labels  = FIELD_LABELS.get(section, {})
    title   = section.replace("_", " ").title()

    # ── Filter out conditional fields that don't apply ─────────────
    def should_show(field):
        if field == "family_doctor_name" and data.get("has_family_doctor") is False:
            return False
        if field == "psychiatrist_name" and data.get("has_psychiatrist") is False:
            return False
        return True

    visible = {f: l for f, l in labels.items() if should_show(f)}

    total  = len(visible)
    filled = sum(1 for f in visible if data.get(f) is not None)
    pct    = int((filled / total) * 100) if total else 0

    fields_html = ""
    for field, label in visible.items():
        value = data.get(field)
        if value is None:
            val_html = '<div style="font-size:13px;color:#cbd5e0;font-style:italic;">—</div>'
        elif isinstance(value, bool):
            color    = "#0F6E56" if value else "#718096"
            bg       = "#e6f4ef" if value else "#f1f5f9"
            text     = "Yes" if value else "No"
            val_html = f'<span style="font-size:12px;color:{color};background:{bg};font-weight:600;padding:2px 10px;border-radius:999px;">{text}</span>'
        else:
            val_html = f'<div style="font-size:14px;color:#1e293b;font-weight:500;">{value}</div>'

        fields_html += f"""
        <div style="padding:11px 0;border-bottom:1px solid #e8f0fb;">
            <div style="font-size:10px;color:#94a3b8;letter-spacing:0.6px;
                        text-transform:uppercase;margin-bottom:5px;">{label}</div>
            {val_html}
        </div>
        """

    return f"""
    <div style="font-family:'Inter',sans-serif;padding:4px 8px;">
        <div style="display:flex;align-items:center;justify-content:space-between;
                    margin-bottom:12px;padding-bottom:12px;border-bottom:2px solid #dbeafe;">
            <div style="font-size:13px;font-weight:600;color:#1e40af;">{title}</div>
            <div style="font-size:11px;color:#94a3b8;">{filled}/{total}</div>
        </div>
        <div style="background:#e2e8f0;border-radius:999px;height:5px;margin-bottom:16px;">
            <div style="background:linear-gradient(90deg,#185FA5,#3b82f6);height:5px;
                        border-radius:999px;width:{pct}%;"></div>
        </div>
        {fields_html}
    </div>
    """

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&display=swap');

*, body, .gradio-container {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
}

body, .gradio-container {
    background: #dce8f5 !important;
}

.gradio-container {
    max-width: 1150px !important;
    margin: 0 auto !important;
    padding: 24px !important;
}

.alice-header {
    display: flex;
    align-items: center;
    gap: 16px;
    background: white;
    border: 1px solid #c5d8f0;
    border-radius: 14px;
    padding: 20px 26px;
    margin-bottom: 16px;
}

.alice-header h1 {
    font-size: 22px !important;
    font-weight: 600 !important;
    color: #1e293b !important;
    margin: 0 !important;
    padding: 0 !important;
    font-family: 'Inter', sans-serif !important;
}

.alice-header p {
    font-size: 13px !important;
    color: #64748b !important;
    margin: 3px 0 0 !important;
}

.btn-general button, .btn-concerns button {
    width: 100% !important;
    padding: 14px 20px !important;
    border-radius: 10px !important;
    font-size: 14px !important;
    font-weight: 500 !important;
    background: white !important;
    color: #1e293b !important;
    text-align: left !important;
    box-shadow: none !important;
    font-family: 'Inter', sans-serif !important;
    transition: background 0.15s ease !important;
}

.btn-general button {
    border: 1px solid #c5d8f0 !important;
    border-left: 3px solid #185FA5 !important;
}

.btn-concerns button {
    border: 1px solid #c5d8f0 !important;
    border-left: 3px solid #0F6E56 !important;
}

.btn-general button:hover, .btn-concerns button:hover {
    background: #f0f6ff !important;
}

.info-box {
    background: #f0f6ff !important;
    border: 1px solid #c5d8f0 !important;
    border-radius: 14px !important;
    padding: 18px !important;
}
"""

def get_fresh_state():
    return copy.deepcopy(initial_state)

def start():
    state           = get_fresh_state()
    welcome_message = [{
        "role":    "assistant",
        "content": "👋 Welcome! Please select a section above to begin your intake."
    }]
    return welcome_message, state, format_display(state)

def chat(message, history, alice_state):
    if alice_state is None:
        _, alice_state, _ = start()
    if alice_state["current_section"] is None:
        return history, alice_state, format_display(alice_state)
    if not message.strip():
        return history, alice_state, format_display(alice_state)

    alice_state["messages"] = alice_state["messages"] + [{"role": "user", "content": message}]
    alice_state = graph.invoke(alice_state)

    reply   = alice_state["messages"][-1]["content"]
    history = history + [
        {"role": "user",      "content": message},
        {"role": "assistant", "content": reply},
    ]
    return history, alice_state, format_display(alice_state)

def switch_section(section, alice_state):
    if alice_state is None:
        _, alice_state, _ = start()

    alice_state["current_section"] = section
    alice_state["messages"]        = []
    alice_state = graph.invoke(alice_state)

    reply   = alice_state["messages"][-1]["content"]
    history = [{"role": "assistant", "content": reply}]
    return history, alice_state, format_display(alice_state)


with gr.Blocks(title="ALICE", css=custom_css, theme=gr.themes.Soft()) as demo:

    alice_state = gr.State(None)

    gr.HTML("""
        <div class="alice-header">
            <div style="width:52px;height:52px;border-radius:50%;background:#dbeafe;
                        display:flex;align-items:center;justify-content:center;
                        font-size:26px;flex-shrink:0;">👩‍⚕️</div>
            <div>
                <h1>Alice</h1>
                <p>Clinical Intake Assistant</p>
            </div>
        </div>
    """)

    with gr.Row():
        btn_general  = gr.Button("General Information", elem_classes="btn-general")
        btn_concerns = gr.Button("Current Concerns",    elem_classes="btn-concerns")

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                height=520,
                show_label=False,
                avatar_images=(USER_AVATAR, ALICE_AVATAR),
            )
            msg = gr.Textbox(
                placeholder="Type your message here...",
                show_label=False,
                container=False,
            )

        with gr.Column(scale=1):
            info_display = gr.HTML(
                value=format_display(None),
                elem_classes="info-box",
                label="Collected Information",
            )

    msg.submit(
        chat,
        inputs=[msg, chatbot, alice_state],
        outputs=[chatbot, alice_state, info_display]
    ).then(lambda: "", outputs=msg)

    btn_general.click(
        lambda s: switch_section("general_information", s),
        inputs=[alice_state],
        outputs=[chatbot, alice_state, info_display]
    )

    btn_concerns.click(
        lambda s: switch_section("current_concerns", s),
        inputs=[alice_state],
        outputs=[chatbot, alice_state, info_display]
    )

    demo.load(start, inputs=[], outputs=[chatbot, alice_state, info_display])

demo.launch()

/var/folders/v9/yz64p8z5579c6d6yx28446qh0000gn/T/ipykernel_59940/4243836903.py:225: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="ALICE", css=custom_css, theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
